# Module 315 — Model Extraction Study

This notebook follows HTB Academy's **Model Reverse Engineering** example. It demonstrates how an exposed prediction API can be queried to collect input/label pairs and train a surrogate classifier that imitates the original.

**Learning flow:** sample plausible inputs → query the original model → fit a surrogate → (optionally) ask the lab to evaluate it.

The API endpoint is intentionally a placeholder. Fill in the active HTB lab address and port when the lesson target is running. Requests go only to the configured endpoint. The final `/model` upload is guarded by `SUBMIT_SURROGATE = False`.

This is a bounded educational lab reproduction. Do not use it against systems outside your authorized HTB lab.


## 1. The idea and the notation

Let `xᵢ` (**“x sub i”**) be the i-th input row, such as one penguin's flipper length and body mass. Let `F(xᵢ)` (**“F of x sub i”**) be the original model's returned label. Let `gθ(xᵢ)` (**“g theta of x sub i”**) be the surrogate's label; `θ` (**“theta”**) represents the surrogate's learned parameters.

The training goal can be written:

`θ* = arg minθ Σᵢ loss(gθ(xᵢ), F(xᵢ))`

Read it aloud: **“theta star is the value of theta that minimizes the sum over i of the loss between g theta of x sub i and F of x sub i.”** `Σᵢ` (“sum over i”) adds the disagreement across examples; `arg min` (“the argument that minimizes”) means choose the parameter setting that makes that total loss smallest.

In plain English: collect examples of what the original says, then adjust the surrogate until its answers resemble those labels. The copy need not use the same internal architecture.

For evaluation, accuracy is `matching predictions ÷ all evaluation examples`. If the surrogate matches the original on 98 of 100 held-out examples, agreement accuracy is `98 / 100 = 0.98`, or 98%. High agreement on training rows alone is not enough to show the surrogate generalizes.


## 2. Imports and lab settings

Keep the sampling budget small and explicit. The sample ranges mirror the lesson's plausible penguin measurements. Set the target URL to the active lab's HTTP address, including its assigned port.

`SUBMIT_SURROGATE` stays off until you deliberately choose to send the trained file to the lab's `/model` evaluation endpoint. That upload is not needed to understand sampling or training.


In [ ]:
import json
import random
import time
from pathlib import Path

import joblib
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from urllib3.util.retry import Retry

# Replace this with the active, authorized HTB lab address and port.
# Example shape only: CLASSIFIER_URL = "http://<TARGET_IP>:<PORT>/"
CLASSIFIER_URL = "http://<TARGET_IP>:<PORT>/"

# Keep the lesson's request volume small and predictable.
N_SAMPLES = 100
RANDOM_SEED = 42
REQUEST_TIMEOUT_SECONDS = 10
DELAY_BETWEEN_REQUESTS_SECONDS = 0.05

# The lesson evaluates the uploaded model at POST /model. Leave this False
# until you intentionally want to transmit the surrogate to the active lab.
SUBMIT_SURROGATE = False

# Check the placeholder before any network request can run.
if "<TARGET_IP>" in CLASSIFIER_URL or "<PORT>" in CLASSIFIER_URL:
    print("Set CLASSIFIER_URL to the active HTB lab URL before querying.")
else:
    print(f"Configured lab endpoint: {CLASSIFIER_URL}")
print(f"Query budget: at most {N_SAMPLES} training requests.")


## 3. Sample plausible measurements

The lesson uses a uniform distribution inside chosen ranges: flipper length 150–250 mm and body mass 2,500–6,500 g. Here `Uniform(a, b)` is read **“uniform from a to b”**: values are drawn across that interval with equal density.

These ranges act like a small domain model. Sampling realistic inputs tends to spend the query budget on examples the classifier was designed to handle. The seed makes this demonstration repeatable.


In [ ]:
rng = random.Random(RANDOM_SEED)

samples_df = pd.DataFrame({
    # random.uniform(a, b) draws one value between the inclusive bounds.
    "Flipper Length (mm)": [rng.uniform(150, 250) for _ in range(N_SAMPLES)],
    "Body Mass (g)": [rng.uniform(2500, 6500) for _ in range(N_SAMPLES)],
})

# These are the two numeric features the surrogate will learn from.
samples_df.head()


## 4. Query the original classifier

For each sampled row, send the feature values as GET parameters. The API's `result` field becomes the label for that row. The response labels are the substitute training targets: we do not need the original training dataset.

This cell stops if the URL is still a placeholder, checks HTTP status, uses a timeout, and pauses briefly between requests. It performs at most `N_SAMPLES` queries. If the service returns errors or a response format different from the lesson, inspect the lab instructions before changing the parsing logic.


In [ ]:
def make_session():
    """Create a session with conservative retries for transient server errors."""
    session = requests.Session()
    retry_policy = Retry(
        total=2,
        backoff_factor=0.5,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=frozenset({"GET"}),
        respect_retry_after_header=True,
    )
    adapter = HTTPAdapter(max_retries=retry_policy)
    session.mount("http://", adapter)
    session.mount("https://", adapter)
    return session

if "<TARGET_IP>" in CLASSIFIER_URL or "<PORT>" in CLASSIFIER_URL:
    raise ValueError("Set CLASSIFIER_URL to the running HTB lab before making requests.")

labels = []
with make_session() as session:
    for row_number, row in samples_df.iterrows():
        # Match the API parameter names shown in the HTB lesson.
        params = {
            "flipper_length": row["Flipper Length (mm)"],
            "body_mass": row["Body Mass (g)"],
        }
        response = session.get(
            CLASSIFIER_URL,
            params=params,
            timeout=REQUEST_TIMEOUT_SECONDS,
        )
        response.raise_for_status()
        result = response.json()
        if "result" not in result:
            raise ValueError(f"Unexpected API response for row {row_number}: {result}")
        labels.append(result["result"])

        # A small pause avoids firing the whole query batch at once.
        if row_number + 1 < len(samples_df):
            time.sleep(DELAY_BETWEEN_REQUESTS_SECONDS)

predictions_df = pd.DataFrame({"species": labels})
print(f"Collected {len(predictions_df)} labels.")
predictions_df.head()


## 5. Train the surrogate classifier

HTB uses logistic regression for this binary classification example. The `StandardScaler` normalizes the two differently scaled measurements before logistic regression fits a separating boundary. `make_pipeline` keeps scaling and classification together so future predictions receive the same preprocessing.

The labels here come from the original API. So fitting `surrogate_model` approximates the original model's behavior; it does not recover the original model's exact weights or architecture.


In [ ]:
# Pipeline: standardize features, then fit a binary logistic-regression model.
surrogate_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(random_state=RANDOM_SEED, max_iter=1000),
)

# X is the sampled measurements; y is the original classifier's answer per row.
surrogate_model.fit(samples_df, predictions_df["species"])

# Save the complete pipeline so its scaler is preserved with the classifier.
MODEL_PATH = Path("surrogate.joblib")
joblib.dump(surrogate_model, MODEL_PATH)
print(f"Saved surrogate pipeline to {MODEL_PATH.resolve()}")


## 6. Optional lab evaluation upload

The HTB lesson sends the serialized model to the lab's `/model` endpoint and receives an accuracy result. This is a separate action from querying and local training, so this cell is explicitly gated. It uploads only when `SUBMIT_SURROGATE` is changed to `True`.

The lab's score is an agreement check against its evaluation set. It does not establish that the surrogate is identical to the original or that it behaves similarly outside the tested domain.


In [ ]:
if not SUBMIT_SURROGATE:
    print("Upload disabled. Set SUBMIT_SURROGATE = True only for the active authorized HTB lab.")
else:
    model_bytes = MODEL_PATH.read_bytes()
    evaluation_url = CLASSIFIER_URL.rstrip("/") + "/model"

    with make_session() as session:
        response = session.post(
            evaluation_url,
            files={"file": (MODEL_PATH.name, model_bytes, "application/octet-stream")},
            timeout=REQUEST_TIMEOUT_SECONDS,
        )
        response.raise_for_status()
        evaluation_result = response.json()

    print(json.dumps(evaluation_result, indent=2))


## 7. Security takeaways and limits

- This is **model extraction**: collect model outputs and train a behavioral copy. It is not the same objective as model inversion, which seeks sensitive information about training data.
- A surrogate can be useful without matching the original architecture. Matching outputs over a useful input distribution is the goal.
- Query range and quality matter. Unrealistic or narrow samples may produce a poor copy even when the code runs correctly.
- Rate limits, monitoring, access controls, and limiting unnecessary output detail can slow or detect extraction. Rate limits have a usability tradeoff and are not a guarantee by themselves.
- The lesson's 100 samples and simple two-feature classifier are a teaching example, not a realistic estimate of effort against a complex production model.

**COAE scenario prompt:** If a service's API returns labels for arbitrary inputs and an outsider trains a similar classifier from those responses, name the attack, the exposed asset, one observable signal, and one defense with its usability tradeoff.


## 8. Sponge examples: token count and denial of ML service

A **denial of ML service (DoML)** attack targets availability by making inference consume too much time, energy, or paid compute. A **sponge example** is an input crafted to increase that work while keeping input size bounded. This is different from simply sending a huge file or flooding the network.

One text-model factor is the number of tokens. A token is a chunk chosen by a tokenizer; it is not always one word or one character. Common text may map efficiently to a few tokens, while rare words or unusual character sequences may split into more pieces. More input tokens can mean more model work. A generative model may also spend more work when it returns a longer output.

The conceptual search objective is `x* = arg max latency(x), subject to input_length(x) ≤ 50`. Read it as **“x star is the input, among inputs no longer than 50 characters, that produces the greatest latency.”** `arg max` means “the input that maximizes”; the length condition limits the search.

HTB describes genetic search: initialize candidate inputs, measure their latency/energy as a **fitness** score, select the highest-cost candidates, combine candidates (**crossover**), make small random edits (**mutation**), and repeat. This notebook does not implement that search and does not send candidate inputs to a service. It only compares tokenizer behavior locally.


### 8.1 Compare character length with token count

HTB demonstrates this with GPT-2's tokenizer. The next cell loads only the tokenizer configuration/vocabulary through Hugging Face Transformers; it does not load GPT-2 model weights or call an inference API. If needed, install `transformers` in the AI environment and allow it to retrieve the tokenizer files for the model linked from the lesson.

The three example strings are: a common phrase, an uncommon long word, and a deliberately unusual character pattern. The exact token pieces are tokenizer-specific, so compare them with the same tokenizer.


In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "openai-community/gpt2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

examples = [
    "This is an example text",
    "Athazagoraphobia",
    "A/h/z/g/r/p/p/",
]

for text in examples:
    tokens = tokenizer.tokenize(text)
    print(f"Text: {text!r}")
    print(f"Characters: {len(text)} | Tokens: {len(tokens)}")
    print(f"Token pieces: {tokens}")
    print()


What to notice: tokenization is a learned representation, so two inputs with similar character counts can create different token counts. This illustrates one possible source of extra work; it does not mean every tokenizer, model, or unusual string will have the same cost pattern. In a lab or production system, measure latency safely and follow the service's authorized limits.


### 8.2 Read the research measurements

The lesson's cited GPU table reports natural, random, and sponge inputs. We calculate simple ratios so the magnitude is easier to see. Ratios compare those measurements only; they are not predictions for a different model or machine.


In [ ]:
# Values transcribed from the HTB lesson's cited GPU experiment.
energy_mj = {"natural": 9492, "random": 25773, "sponge": 40976}
latency_ms = {"natural": 0.10, "random": 0.24, "sponge": 0.37}

print(f"Sponge / natural energy: {energy_mj['sponge'] / energy_mj['natural']:.2f}x")
print(f"Sponge / natural latency: {latency_ms['sponge'] / latency_ms['natural']:.2f}x")

# Separate black-box example in the lesson: about 1 ms to about 6 seconds.
ordinary_ms = 1
sponge_ms = 6 * 1000  # convert seconds to milliseconds
print(f"Azure example ratio: {sponge_ms / ordinary_ms:.0f}x")


### COAE recall: identify impact and controls

For a suspected sponge-example DoML scenario, ask: Which resource is consumed? Can the input stay within normal size limits yet drive high latency or energy? Does the attacker need a local copy of the model, or only query access and timing?

Controls can include rate limits and anomaly detection, bounded input/output token budgets, concurrency limits, and maximum inference time or energy. A threshold that is too strict can reject legitimate workloads, so choose limits from measurements of normal usage. This notebook deliberately stops at tokenization and measurement interpretation; it does not stress a live model.
